# Pipeline — Regresión Lineal
### Módulo 3 · Tema 3.1

---
## ¿Qué hace la Regresión Lineal?
Predice un **número continuo** (no una categoría).
Busca la línea (o hiperplano) que mejor se ajusta a los datos.

```
ŷ = w₀ + w₁·x₁ + w₂·x₂ + ... + wₙ·xₙ
```
- **ŷ** = la predicción (un número)
- **w₀** = intercepto (bias, el valor cuando todo es 0)
- **w₁...wₙ** = pesos que el modelo aprende
- **x₁...xₙ** = las características de entrada

## ¿Cuándo usarla?
- Predecir precios de casas
- Predecir temperatura
- Predecir ventas
- Cualquier caso donde la salida sea un **número real** sin límite

## Métricas de evaluación
```
MSE  = promedio de (real - predicho)²    ← penaliza errores grandes
RMSE = √MSE                              ← mismas unidades que Y
R²   = qué % de la variación explica     ← 1.0=perfecto, 0=nada
```

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA 1 — Imports
# ═══════════════════════════════════════════════════════════════
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model  import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics       import mean_squared_error, r2_score

# Ruta relativa al machote (ajusta los '..' según tu estructura)
RUTA = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'Machote'))
if RUTA not in sys.path:
    sys.path.append(RUTA)

print("Imports listos ✅")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA 2 — Cargar datos
# Usamos 'diabetes' porque su Y es un número continuo
# ← CAMBIA ESTE BLOQUE por tu propio dataset si aplica
# ═══════════════════════════════════════════════════════════════
from sklearn.datasets import load_diabetes

dataset = load_diabetes()
X = pd.DataFrame(dataset.data, columns=dataset.feature_names)
Y = dataset.target   # número continuo (nivel de diabetes un año después)

print(f"Muestras: {X.shape[0]}  |  Características: {X.shape[1]}")
print(f"Y — min: {Y.min():.1f}, max: {Y.max():.1f}, media: {Y.mean():.1f}")
X.head()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA 3 — Exploración rápida
# ═══════════════════════════════════════════════════════════════
# Correlación de cada característica con Y
# Alta correlación (|valor| > 0.3) = esa columna ayuda a predecir
df_temp = X.copy()
df_temp['Y'] = Y
correlaciones = df_temp.corr()['Y'].drop('Y').sort_values(key=abs, ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Gráfica 1: correlación con Y
colors = ['#D85A30' if abs(v) >= 0.3 else '#aaa' for v in correlaciones]
correlaciones.plot(kind='barh', ax=axes[0], color=colors)
axes[0].axvline(x=0, color='gray', linewidth=0.8)
axes[0].set_title('Correlación con Y')

# Gráfica 2: distribución de Y
axes[1].hist(Y, bins=25, color='#3B8BD4', edgecolor='white')
axes[1].set_title('Distribución de Y (lo que predecimos)')
axes[1].set_xlabel('Valor de Y')
axes[1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

print("Top 5 características más correlacionadas con Y:")
print(correlaciones.head().to_string())

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA 4 — Preparar datos
# ═══════════════════════════════════════════════════════════════
# Dividir en train y test
# stratify=None porque Y es continuo (no hay clases que balancear)
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.25, random_state=42
)

# Escalar — importante para que columnas con números grandes
# no dominen sobre columnas con números pequeños
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)  # aprende la escala del train
X_test_sc  = scaler.transform(X_test)        # aplica esa misma escala al test

print(f"Train: {len(X_train)} muestras  |  Test: {len(X_test)} muestras")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA 5 — Crear y entrenar el modelo
# ═══════════════════════════════════════════════════════════════
modelo = LinearRegression()
# .fit() = entrenar. El modelo busca los pesos que minimizan el MSE
modelo.fit(X_train_sc, y_train)

print("Modelo entrenado ✅")
print(f"\nPesos aprendidos (primeros 5):")
for feat, peso in zip(X.columns[:5], modelo.coef_[:5]):
    print(f"  {feat:<20} w = {peso:+.4f}")
print(f"  Intercepto (bias): {modelo.intercept_:.4f}")
print()
print("Interpretación: un peso positivo significa que cuando")
print("esa característica sube, la predicción también sube.")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA 6 — Predecir y evaluar
# ═══════════════════════════════════════════════════════════════
Y_pred = modelo.predict(X_test_sc)

mse  = mean_squared_error(y_test, Y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, Y_pred)

print("═" * 50)
print("  RESULTADOS — Regresión Lineal")
print("═" * 50)
print(f"  MSE:  {mse:.2f}")
print(f"  RMSE: {rmse:.2f}  ← el modelo se equivoca aprox. ±{rmse:.1f} unidades")
print(f"  R²:   {r2:.4f}  ← explica el {r2*100:.1f}% de la variación")

# Gráfica real vs predicho
# Si el modelo fuera perfecto, todos los puntos estarían en la línea roja
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(y_test, Y_pred, alpha=0.5, s=20, color='#3B8BD4')
lims = [min(y_test.min(), Y_pred.min()), max(y_test.max(), Y_pred.max())]
ax.plot(lims, lims, 'r--', linewidth=1.5, label='Predicción perfecta')
ax.set_xlabel('Valores reales')
ax.set_ylabel('Valores predichos')
ax.set_title(f'Real vs Predicho — R²={r2:.3f}')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA 7 — ¿Cómo mejorar el modelo? (opcional)
# ═══════════════════════════════════════════════════════════════
# Probar seleccionando solo las mejores características

top_features = correlaciones.abs().head(5).index.tolist()
X_top = X[top_features]

X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X_top, Y, test_size=0.25, random_state=42)
sc2 = StandardScaler()
X_tr2_sc = sc2.fit_transform(X_tr2)
X_te2_sc  = sc2.transform(X_te2)

modelo2 = LinearRegression()
modelo2.fit(X_tr2_sc, y_tr2)
Y_pred2 = modelo2.predict(X_te2_sc)

r2_top = r2_score(y_te2, Y_pred2)
print(f"R² con TODAS las características ({X.shape[1]}): {r2:.4f}")
print(f"R² con TOP 5 características:                  {r2_top:.4f}")
print()
print("Si el R² sube con menos columnas → las otras eran ruido.")
print("Si el R² baja → las columnas eliminadas sí aportaban.")